# 06 — Por qué son atractivos: las variables detrás del ranking

**Proyecto:** RADAR Cibest  
**Propósito:** mostrar los datos y scores dimensionales que explican las posiciones
del ranking. Un score sin contexto es un número; las dimensiones y variables son el argumento.

In [22]:
from __future__ import annotations
import sys, warnings, importlib, re
from pathlib import Path
from typing import Dict, List
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import HTML, display, Markdown

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent))

# ---------------------------------------------------------------------------
# Reload exhaustivo de todos los módulos del pipeline para evitar caché stale
# ---------------------------------------------------------------------------
import src.utils
import src.data_preparation.cleaning
import src.data_preparation.feature_engineering
import src.data_extraction.complementary
import src.scoring.weighting
import src.scoring.ranking
import src.scoring.gravity
import src.scoring.hybrid_scorer

for mod in [
    src.utils,
    src.data_preparation.cleaning,
    src.data_preparation.feature_engineering,
    src.data_extraction.complementary,
    src.scoring.weighting,
    src.scoring.ranking,
    src.scoring.gravity,
    src.scoring.hybrid_scorer,
]:
    importlib.reload(mod)

from src.utils import load_all_configs, resolve_data_path, get_variable_catalog
from src.scoring.hybrid_scorer import prepare_decision_matrix, run_full_scoring

configs = load_all_configs()
variable_catalog = get_variable_catalog(configs["variables"])

In [23]:
# --- Paleta Cibest ---
C = {
    "black": "#1A1A1A", "white": "#FFFFFF", "gold": "#FDD923",
    "gold_light": "#FFF7D3", "gray": "#2C2A28", "gray_mid": "#666666",
    "gray_light": "#E5E5E5", "green": "#0BA682", "red": "#C62828", "amber": "#F39C12",
}

def insight_box(title, text):
    display(HTML(f'<div style="border-left:6px solid {C["gold"]}; background-color:{C["gold_light"]}; padding:14px 18px; margin:12px 0; font-family:Arial; color:{C["gray"]};"><b>{title}</b><br>{text}</div>'))

def style_table(df, gradient_cols=None, cmap="RdYlGn", fmt=None):
    s = df.style.set_table_styles([
        {"selector": "th", "props": [("background-color", C["gray"]), ("color", C["gold"]),
            ("font-weight", "bold"), ("text-align", "center"), ("padding", "8px"), ("font-family", "Arial")]},
        {"selector": "td", "props": [("padding", "6px 10px"), ("font-family", "Arial")]},
    ])
    if gradient_cols: s = s.background_gradient(subset=gradient_cols, cmap=cmap)
    if fmt: s = s.format(fmt)
    return s

In [24]:
# ---------------------------------------------------------------------------
# 1. Carga autocontenida: master → scoring completo → wide_raw + rankings
# ---------------------------------------------------------------------------
raw_dir = resolve_data_path(configs["settings"]["data"]["raw_path"])
pattern = re.compile(r"^master_raw_\d{8}\.parquet$")

candidates = sorted(
    [path for path in raw_dir.glob("master_raw_*.parquet") if pattern.match(path.name)],
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)

if not candidates:
    raise FileNotFoundError("No se encontró master_raw_YYYYMMDD.parquet. Ejecute primero el notebook 01.")

master_path = candidates[0]
master = pd.read_parquet(master_path)

required_cols = {"country_iso3", "year", "variable", "value", "source"}
missing_cols = required_cols - set(master.columns)
if missing_cols:
    raise ValueError(f"Master inválido. Faltan columnas requeridas: {sorted(missing_cols)}")

master = master.copy()
master["country_iso3"] = master["country_iso3"].astype(str).str.strip()
master["variable"] = master["variable"].astype(str).str.strip()
master["year"] = pd.to_numeric(master["year"], errors="coerce")
master["value"] = pd.to_numeric(master["value"], errors="coerce")

summary_master = pd.DataFrame({
    "métrica": ["archivo", "filas", "países", "variables", "fuentes", "año_min", "año_max", "incluye_gdp_growth"],
    "valor": [
        master_path.name,
        master.shape[0],
        master["country_iso3"].nunique(),
        master["variable"].nunique(),
        master["source"].nunique(),
        int(master["year"].min()),
        int(master["year"].max()),
        "Sí" if "gdp_growth" in master["variable"].unique() else "No",
    ],
})

display(style_table(summary_master))

results = run_full_scoring(master, configs, persist=False)

# wide_raw: valores absolutos por país (está en results, no en disco)
wide_raw = results["wide_raw"].copy()
if "country_iso3" not in wide_raw.columns:
    wide_raw = wide_raw.reset_index().rename(columns={"index": "country_iso3"})

# global_ranking: scores TOPSIS global con scores dimensionales
global_ranking = results["global_ranking"].copy()
if "country_iso3" not in global_ranking.columns:
    global_ranking = global_ranking.reset_index().rename(columns={"index": "country_iso3"})

# RADAR global
radar_global = results["radar_global"].copy()
if "country_iso3" not in radar_global.columns:
    radar_global = radar_global.reset_index().rename(columns={"index": "country_iso3"})
radar_global = radar_global.sort_values("radar_score", ascending=False)
if "rank" not in radar_global.columns:
    radar_global["rank"] = radar_global["radar_score"].rank(ascending=False, method="min").astype(int)

top_15 = radar_global.head(15)["country_iso3"].tolist()
top_10 = top_15[:10]
top_5 = top_15[:5]

print(f"Wide raw: {wide_raw.shape[0]} países × {wide_raw.shape[1]} columnas")
print(f"Variables disponibles: {[c for c in wide_raw.columns if c != 'country_iso3']}")
print(f"Top 5 RADAR: {', '.join(top_5)}")

,métrica,valor
0,archivo,master_raw_20260624.parquet
1,filas,1288
2,países,30
3,variables,45
4,fuentes,3
5,año_min,1996
6,año_max,2026
7,incluye_gdp_growth,Sí


2026-06-25 10:15:24.265 | INFO     | src.data_preparation.cleaning:pivot_latest_value_and_year:133 - Pivoteo ancho: 30 paises x 45 variables (estrategia=latest_available)
2026-06-25 10:15:24.274 | INFO     | src.data_preparation.cleaning:apply_freshness_filter:200 - Control de antigüedad aplicado: reference_year=2026, max_age=6, celdas stale=51
2026-06-25 10:15:24.277 | INFO     | src.data_preparation.cleaning:apply_freshness_filter:225 - Variables con más datos stale: {'public_debt_gdp': 8, 'bank_capital_rwa': 6, 'bank_concentration_5': 5, 'stock_market_cap_gdp': 5, 'financial_system_deposits_gdp': 4, 'interest_rate_spread': 4, 'atms_per_100k_adults': 3, 'current_account_gdp': 2, 'ict_goods_exports_pct_total_goods_exports': 2, 'domestic_credit_private_gdp': 2, 'digital_payment_adults_pct': 2, 'personal_remittances_gdp': 2, 'trade_openness': 2, 'commercial_bank_branches_per_100k_adults': 1, 'gdp_per_capita_ppp': 1}
2026-06-25 10:15:24.278 | INFO     | src.data_preparation.cleaning:appl

Wide raw: 24 países × 47 columnas
Variables disponibles: ['gdp_nominal', 'gdp_per_capita_ppp', 'gdp_growth', 'inflation_rate', 'population_total', 'urban_population_pct', 'unemployment_rate', 'current_account_gdp', 'public_debt_gdp', 'trade_openness', 'fdi_net_inflows_gdp', 'weighted_mean_applied_tariff_all_products', 'domestic_credit_private_gdp', 'account_ownership', 'interest_rate_spread', 'bank_npl_ratio', 'stock_market_cap_gdp', 'personal_remittances_gdp', 'bank_concentration_5', 'financial_system_deposits_gdp', 'bank_capital_rwa', 'regulatory_quality', 'government_effectiveness', 'rule_of_law', 'political_stability', 'voice_accountability', 'control_of_corruption', 'country_risk_premium', 'heritage_efi', 'internet_users_pct', 'mobile_subscriptions', 'digital_payment_adults_pct', 'secure_internet_servers_per_million', 'commercial_bank_branches_per_100k_adults', 'atms_per_100k_adults', 'ict_goods_exports_pct_total_goods_exports', 'geographic_distance_km', 'common_language_spanish',

In [25]:
# ---------------------------------------------------------------------------
# 2. Ejecutar scoring completo
# ---------------------------------------------------------------------------
print("\nEjecutando run_full_scoring()...")
results = run_full_scoring(master, configs, persist=False)
print("  ✓ Scoring completado")

2026-06-25 10:15:24.735 | INFO     | src.data_preparation.cleaning:pivot_latest_value_and_year:133 - Pivoteo ancho: 30 paises x 45 variables (estrategia=latest_available)


2026-06-25 10:15:24.745 | INFO     | src.data_preparation.cleaning:apply_freshness_filter:200 - Control de antigüedad aplicado: reference_year=2026, max_age=6, celdas stale=51
2026-06-25 10:15:24.750 | INFO     | src.data_preparation.cleaning:apply_freshness_filter:225 - Variables con más datos stale: {'public_debt_gdp': 8, 'bank_capital_rwa': 6, 'bank_concentration_5': 5, 'stock_market_cap_gdp': 5, 'financial_system_deposits_gdp': 4, 'interest_rate_spread': 4, 'atms_per_100k_adults': 3, 'current_account_gdp': 2, 'ict_goods_exports_pct_total_goods_exports': 2, 'domestic_credit_private_gdp': 2, 'digital_payment_adults_pct': 2, 'personal_remittances_gdp': 2, 'trade_openness': 2, 'commercial_bank_branches_per_100k_adults': 1, 'gdp_per_capita_ppp': 1}
2026-06-25 10:15:24.754 | INFO     | src.data_preparation.cleaning:apply_freshness_filter:230 - Países con más datos stale: {'VEN': 12, 'BRB': 5, 'TTO': 4, 'CAN': 3, 'HTI': 3, 'BOL': 2, 'BLZ': 2, 'CHL': 2, 'CUB': 2, 'DOM': 2, 'ECU': 2, 'SUR':


Ejecutando run_full_scoring()...


2026-06-25 10:15:25.017 | INFO     | src.data_preparation.cleaning:impute_missing:335 - Imputacion (regional_median): 91 -> 0 celdas faltantes
2026-06-25 10:15:25.024 | INFO     | src.data_preparation.cleaning:run_cleaning:454 - Limpieza completada: 25 países x 45 variables | excluidos=['BRB', 'CUB', 'GUY', 'HTI', 'VEN']
2026-06-25 10:15:25.039 | INFO     | src.data_preparation.feature_engineering:apply_log_transform:73 - Log-transformacion (natural) aplicada a: ['gdp_nominal', 'population_total', 'geographic_distance_km', 'colombian_diaspora_stock', 'stock_market_cap_gdp', 'financial_system_deposits_gdp', 'domestic_credit_private_gdp', 'fdi_net_inflows_gdp', 'personal_remittances_gdp', 'secure_internet_servers_per_million', 'atms_per_100k_adults', 'commercial_bank_branches_per_100k_adults']
2026-06-25 10:15:25.050 | INFO     | src.data_preparation.feature_engineering:run_feature_engineering:136 - Feature engineering completado: 25 paises x 46 variables
2026-06-25 10:15:25.054 | INFO  

  ✓ Scoring completado


In [26]:
# ---------------------------------------------------------------------------
# 3. Extraer wide_raw, rankings y scores dimensionales
# ---------------------------------------------------------------------------

# --- wide_raw ---
wide_raw = results["wide_raw"].copy()

# Diagnóstico del índice
print(f"\nwide_raw original:")
print(f"  Shape: {wide_raw.shape}")
print(f"  Index name: {wide_raw.index.name}")
print(f"  Columnas ({len(wide_raw.columns)}): {list(wide_raw.columns[:10])}{'...' if len(wide_raw.columns) > 10 else ''}")

# Normalizar: country_iso3 siempre como columna
if wide_raw.index.name == "country_iso3":
    wide_raw = wide_raw.reset_index()
elif "country_iso3" not in wide_raw.columns and wide_raw.index.dtype == "object":
    wide_raw = wide_raw.reset_index().rename(columns={wide_raw.index.name or "index": "country_iso3"})

# --- global_ranking (scores dimensionales TOPSIS) ---
global_ranking = results["global_ranking"].copy()
if global_ranking.index.name == "country_iso3":
    global_ranking = global_ranking.reset_index()
elif "country_iso3" not in global_ranking.columns:
    global_ranking = global_ranking.reset_index().rename(columns={global_ranking.index.name or "index": "country_iso3"})

# --- RADAR global ---
radar_global = results["radar_global"].copy()
if radar_global.index.name == "country_iso3":
    radar_global = radar_global.reset_index()
elif "country_iso3" not in radar_global.columns:
    radar_global = radar_global.reset_index().rename(columns={radar_global.index.name or "index": "country_iso3"})
radar_global = radar_global.sort_values("radar_score", ascending=False)
if "rank" not in radar_global.columns:
    radar_global["rank"] = radar_global["radar_score"].rank(ascending=False, method="min").astype(int)

top_15 = radar_global.head(15)["country_iso3"].tolist()
top_10 = top_15[:10]
top_5 = top_15[:5]


wide_raw original:
  Shape: (24, 46)
  Index name: country_iso3
  Columnas (46): ['gdp_nominal', 'gdp_per_capita_ppp', 'gdp_growth', 'inflation_rate', 'population_total', 'urban_population_pct', 'unemployment_rate', 'current_account_gdp', 'public_debt_gdp', 'trade_openness']...


In [27]:
# ---------------------------------------------------------------------------
# 4. Inventario de variables disponibles en wide_raw
# ---------------------------------------------------------------------------
available_vars = [c for c in wide_raw.columns if c != "country_iso3"]
print(f"\n{'='*60}")
print(f"INVENTARIO: {len(available_vars)} variables en wide_raw")
print(f"{'='*60}")

# Agrupar por dimensión del catálogo
dim_vars_available: Dict[str, List[str]] = {}
uncatalogued = []
for var in available_vars:
    meta = variable_catalog.get(var, None)
    if meta:
        dim = meta.get("dimension", "sin_dimension")
        dim_vars_available.setdefault(dim, []).append(var)
    else:
        uncatalogued.append(var)

for dim_name, dim_label in [("macro", "D1 Macro"), ("financial", "D2 Financiera"),
                             ("institutional", "D3 Institucional"), ("digital_tech", "D4 Digital-Tech"),
                             ("proximity", "D5 Proximidad")]:
    dim_v = dim_vars_available.get(dim_name, [])
    print(f"\n  {dim_label}: {len(dim_v)} variables")
    for v in dim_v:
        desc = variable_catalog[v].get("description", v)[:50]
        non_null = wide_raw[v].notna().sum()
        print(f"    {v}: {desc} ({non_null} países con dato)")

if uncatalogued:
    print(f"\n  Sin catálogo: {len(uncatalogued)} → {', '.join(uncatalogued[:10])}")

# Scores dimensionales disponibles
dim_score_cols = [c for c in ["score_macro", "score_financial", "score_institutional", "score_digital_tech"]
                  if c in global_ranking.columns]
print(f"\n  Scores dimensionales en global_ranking: {dim_score_cols}")
print(f"\nTop 5 RADAR: {', '.join(top_5)}")


INVENTARIO: 46 variables en wide_raw

  D1 Macro: 12 variables
    gdp_nominal: PIB nominal en USD corrientes (24 países con dato)
    gdp_per_capita_ppp: PIB per capita PPP en dolares internacionales corr (24 países con dato)
    gdp_growth: Crecimiento anual real del PIB (24 países con dato)
    inflation_rate: Inflacion anual de precios al consumidor (24 países con dato)
    population_total: Poblacion total (24 países con dato)
    urban_population_pct: Poblacion urbana como porcentaje del total (24 países con dato)
    unemployment_rate: Tasa de desempleo (%) (24 países con dato)
    current_account_gdp: Cuenta corriente como % del PIB (24 países con dato)
    public_debt_gdp: Deuda pública como % del PIB (24 países con dato)
    trade_openness: Apertura comercial (% del PIB) (24 países con dato)
    fdi_net_inflows_gdp: Inversión extranjera directa (% del PIB) (24 países con dato)
    weighted_mean_applied_tariff_all_products: Tarifa aplicada promedio ponderada para todos los  (

## 2. Radar chart: perfil dimensional de los top-5 (5 dimensiones incluyendo proximidad)

Cada país tiene un score TOPSIS parcial por dimensión estructural (macro, financiera,
institucional, digital-tecnológica) más el índice de proximidad con Colombia (IPC).
Este gráfico revela por qué cada país ocupa su posición en el ranking.

In [28]:
DIM_COLS = ["score_macro", "score_financial", "score_institutional", "score_digital_tech"]
DIM_LABELS_SHORT = ["Macro", "Financiera", "Institucional", "Digital-Tech"]
dim_available = [c for c in DIM_COLS if c in global_ranking.columns]

# Incorporar IPC como 5ª dimensión
ipc_df = results.get("ipc", None)
has_ipc = ipc_df is not None
if has_ipc:
    ipc_series = ipc_df.copy()
    if isinstance(ipc_series, pd.DataFrame):
        if "country_iso3" not in ipc_series.columns:
            ipc_series = ipc_series.reset_index().rename(columns={ipc_series.index.name or "index": "country_iso3"})
        ipc_col = [c for c in ipc_series.columns if c != "country_iso3"][0]
        ipc_map = ipc_series.set_index("country_iso3")[ipc_col].to_dict()
    elif isinstance(ipc_series, pd.Series):
        ipc_map = ipc_series.to_dict()
    else:
        ipc_map = {}
        has_ipc = False

if len(dim_available) >= 3:
    dim_data = global_ranking.set_index("country_iso3")[dim_available].copy()
    if has_ipc:
        dim_data["ipc"] = dim_data.index.map(ipc_map)
        all_dims = dim_available + ["ipc"]
        all_labels = DIM_LABELS_SHORT[:len(dim_available)] + ["Proximidad\nColombia"]
    else:
        all_dims = dim_available
        all_labels = DIM_LABELS_SHORT[:len(dim_available)]

    medians = dim_data[all_dims].median()

    # --- Colores Cibest diferenciados por país ---
    country_styles = {
        top_5[0]: {"color": C["gold"],    "width": 3.5, "dash": "solid",    "fill_opacity": 0.15},
        top_5[1]: {"color": C["green"],   "width": 3.0, "dash": "solid",    "fill_opacity": 0.12},
        top_5[2]: {"color": "#FFFFFF",    "width": 3.0, "dash": "dot",      "fill_opacity": 0.08},
        top_5[3]: {"color": C["amber"],   "width": 2.5, "dash": "dashdot",  "fill_opacity": 0.10},
        top_5[4]: {"color": "#4FC3F7",    "width": 2.5, "dash": "longdash", "fill_opacity": 0.08},
    }

    labels_closed = all_labels + [all_labels[0]]
    fig = go.Figure()

    for country in top_5:
        if country not in dim_data.index:
            continue
        style = country_styles.get(country, {"color": C["gray_mid"], "width": 2, "dash": "solid", "fill_opacity": 0.1})
        vals = dim_data.loc[country, all_dims].tolist()
        vals_closed = vals + [vals[0]]

        # Score RADAR para la leyenda
        r_score = radar_global.loc[radar_global["country_iso3"] == country, "radar_score"]
        r_label = f"{country} (RADAR {r_score.values[0]:.3f})" if len(r_score) > 0 else country

        fig.add_trace(go.Scatterpolar(
            r=vals_closed, theta=labels_closed, name=r_label,
            line=dict(color=style["color"], width=style["width"], dash=style["dash"]),
            fill="toself", fillcolor=f"rgba({int(style['color'].lstrip('#')[0:2], 16)},{int(style['color'].lstrip('#')[2:4], 16)},{int(style['color'].lstrip('#')[4:6], 16)},{style['fill_opacity']})",
            marker=dict(size=6, color=style["color"]),
        ))

    # Mediana del universo
    med_vals = medians.tolist() + [medians.tolist()[0]]
    fig.add_trace(go.Scatterpolar(
        r=med_vals, theta=labels_closed, name="Mediana universo",
        line=dict(color=C["gray_mid"], width=2, dash="dash"),
        marker=dict(size=0),
        fillcolor="rgba(0,0,0,0)",
    ))

    fig.update_layout(
        title=dict(
            text="Perfil dimensional de los top-5: no todos llegan al ranking por las mismas razones",
            font=dict(size=15),
        ),
        polar=dict(
            bgcolor=C["black"],
            radialaxis=dict(
                range=[0, 1], showticklabels=True, tickformat=".1f",
                gridcolor="rgba(255,255,255,0.15)", tickfont=dict(color=C["gray_mid"], size=10),
            ),
            angularaxis=dict(
                gridcolor="rgba(255,255,255,0.15)",
                tickfont=dict(color=C["white"], size=12, family="Arial"),
            ),
        ),
        paper_bgcolor=C["black"],
        font=dict(family="Arial", color=C["white"]),
        legend=dict(
            font=dict(size=12, color=C["white"]),
            bgcolor="rgba(0,0,0,0.5)",
            bordercolor=C["gray_mid"], borderwidth=1,
        ),
        height=620,
    )
    fig.show()

    insight_box("Lectura del radar",
        "PAN y CRI sobresalen en Proximidad pero quedan por debajo de la mediana en lo estructural. "
        "ESP, CHL mantienen perfil equilibrado en las 5 dimensiones. "
        "La asimetría entre proximidad y estructura es la tensión central del ranking RADAR.")

### Radar chart individual por país del top-5

Vista individual para presentaciones: cada país contra la mediana del universo.

In [29]:
if len(dim_available) >= 3:
    for country in top_5:
        if country not in dim_data.index:
            continue
        r_score = radar_global.loc[radar_global["country_iso3"] == country, "radar_score"]
        r_val = f"{r_score.values[0]:.3f}" if len(r_score) > 0 else "N/A"
        r_rank = radar_global.loc[radar_global["country_iso3"] == country, "rank"]
        r_pos = f"#{int(r_rank.values[0])}" if len(r_rank) > 0 else ""

        vals = dim_data.loc[country, all_dims].tolist() + [dim_data.loc[country, all_dims].tolist()[0]]

        fig = go.Figure()
        fig.add_trace(go.Scatterpolar(
            r=vals, theta=labels_closed, name=country,
            line=dict(color=C["gold"], width=3),
            fill="toself", fillcolor=f"rgba(253,217,35,0.20)",
            marker=dict(size=7, color=C["gold"]),
        ))
        fig.add_trace(go.Scatterpolar(
            r=med_vals, theta=labels_closed, name="Mediana",
            line=dict(color=C["gray_mid"], width=2, dash="dash"),
            marker=dict(size=0),
        ))
        fig.update_layout(
            title=dict(text=f"{country} {r_pos} — Score RADAR: {r_val}", font=dict(size=14)),
            polar=dict(
                bgcolor=C["black"],
                radialaxis=dict(range=[0, 1], tickformat=".1f",
                                gridcolor="rgba(255,255,255,0.15)", tickfont=dict(color=C["gray_mid"])),
                angularaxis=dict(gridcolor="rgba(255,255,255,0.15)",
                                 tickfont=dict(color=C["white"], size=11)),
            ),
            paper_bgcolor=C["black"], font=dict(family="Arial", color=C["white"]),
            legend=dict(font=dict(color=C["white"]), bgcolor="rgba(0,0,0,0)"),
            height=420, showlegend=False,
        )
        fig.show()

In [30]:
# Tabla de scores dimensionales + IPC para top-15
if dim_available:
    dim_table = global_ranking[global_ranking["country_iso3"].isin(top_15)][
        ["country_iso3", "score", "rank"] + dim_available
    ].sort_values("rank").copy()
    if has_ipc:
        dim_table["ipc"] = dim_table["country_iso3"].map(ipc_map)
    dim_table = dim_table.rename(columns={
        "score": "TOPSIS_global", "score_macro": "Macro", "score_financial": "Financiera",
        "score_institutional": "Institucional", "score_digital_tech": "Digital-Tech",
        "ipc": "Proximidad",
    })
    grad_cols = [c for c in ["TOPSIS_global", "Macro", "Financiera", "Institucional", "Digital-Tech", "Proximidad"] if c in dim_table.columns]
    display(style_table(
        dim_table, gradient_cols=grad_cols,
        fmt={c: "{:.3f}" for c in dim_table.columns if dim_table[c].dtype in ["float64", "float32"]},
    ).set_caption("Scores dimensionales TOPSIS + Proximidad — Top 15 RADAR"))

,country_iso3,TOPSIS_global,rank,Macro,Financiera,Institucional,Digital-Tech,Proximidad
0,USA,0.703,1,0.691,0.648,0.786,0.716,0.286
1,CAN,0.693,2,0.685,0.573,0.905,0.634,0.174
2,ESP,0.644,3,0.604,0.604,0.725,0.712,0.000
3,CHL,0.609,4,0.519,0.554,0.763,0.645,0.264
4,URY,0.580,5,0.454,0.493,0.808,0.608,0.189
5,CRI,0.544,6,0.451,0.481,0.709,0.547,0.795
6,BHS,0.528,7,0.386,0.558,0.652,0.506,0.503
7,PAN,0.513,8,0.459,0.528,0.543,0.532,0.951
10,DOM,0.497,11,0.448,0.492,0.559,0.437,0.792
12,MEX,0.487,13,0.592,0.461,0.423,0.448,0.346


## 3. Heatmap dimensional: fortalezas y debilidades de un vistazo

In [31]:
if dim_available:
    heatmap_cols = dim_available.copy()
    heatmap_dim = global_ranking[global_ranking["country_iso3"].isin(top_15)].set_index("country_iso3")[heatmap_cols].copy()
    if has_ipc:
        heatmap_dim["ipc"] = heatmap_dim.index.map(ipc_map)
        heatmap_cols.append("ipc")
    heatmap_dim = heatmap_dim.rename(columns={
        "score_macro": "Macro", "score_financial": "Financiera",
        "score_institutional": "Institucional", "score_digital_tech": "Digital-Tech",
        "ipc": "Proximidad",
    })
    heatmap_dim = heatmap_dim.reindex(top_15)

    fig = px.imshow(
        heatmap_dim, color_continuous_scale="RdYlGn", aspect="auto",
        title="Heatmap dimensional: verde = fortaleza relativa, rojo = debilidad",
        text_auto=".2f",
    )
    fig.update_layout(height=550, xaxis_title="Dimensión", yaxis_title="País",
                      font=dict(family="Arial"))
    fig.show()

## 4. Variables absolutas disponibles en el modelo

Las secciones siguientes muestran las variables reales del wide_raw.
Solo se grafican las variables que existen en el dataset.

In [32]:
# Descubrir variables disponibles y mapearlas a dimensiones
available_vars = [c for c in wide_raw.columns if c != "country_iso3"]
print(f"\n{len(available_vars)} variables disponibles en wide_raw:")

dim_vars_available: Dict[str, List[str]] = {}
for var in available_vars:
    meta = variable_catalog.get(var, {})
    dim = meta.get("dimension", "other")
    dim_vars_available.setdefault(dim, []).append(var)
    
for dim, vars_list in sorted(dim_vars_available.items()):
    print(f"  {dim}: {', '.join(vars_list)}")


46 variables disponibles en wide_raw:
  digital_tech: internet_users_pct, mobile_subscriptions, digital_payment_adults_pct, secure_internet_servers_per_million, commercial_bank_branches_per_100k_adults, atms_per_100k_adults, ict_goods_exports_pct_total_goods_exports
  financial: domestic_credit_private_gdp, account_ownership, interest_rate_spread, bank_npl_ratio, stock_market_cap_gdp, personal_remittances_gdp, bank_concentration_5, financial_system_deposits_gdp, bank_capital_rwa
  institutional: regulatory_quality, government_effectiveness, rule_of_law, political_stability, voice_accountability, control_of_corruption, country_risk_premium, heritage_efi
  macro: gdp_nominal, gdp_per_capita_ppp, gdp_growth, inflation_rate, population_total, urban_population_pct, unemployment_rate, current_account_gdp, public_debt_gdp, trade_openness, fdi_net_inflows_gdp, weighted_mean_applied_tariff_all_products
  other: cultural_distance_hofstede
  proximity: geographic_distance_km, common_language_spa

In [33]:
# Helper genérico: tabla + barplot horizontal de una variable
def barplot_variable(var_name: str, title: str, xlabel: str, reverse_color: bool = False):
    if var_name not in wide_raw.columns:
        display(Markdown(f"*Variable `{var_name}` no disponible en wide_raw.*"))
        return
    plot_df = wide_raw[wide_raw["country_iso3"].isin(top_15)][["country_iso3", var_name]].dropna()
    plot_df = plot_df.sort_values(var_name, ascending=False).reset_index(drop=True)

    # Agregar ranking RADAR para contexto
    plot_df = plot_df.merge(
        radar_global[["country_iso3", "radar_score", "rank"]].rename(columns={"rank": "rank_RADAR"}),
        on="country_iso3", how="left",
    )
    plot_df["rank_variable"] = plot_df[var_name].rank(ascending=(not reverse_color), method="min").astype(int)

    # Tabla
    direction = variable_catalog.get(var_name, {}).get("direction", "positive")
    dir_label = "↓ menor = mejor" if direction == "negative" else "↑ mayor = mejor"
    display(Markdown(f"**{title}** ({dir_label})"))
    display(style_table(
        plot_df[["country_iso3", var_name, "rank_variable", "rank_RADAR"]].rename(columns={
            var_name: "Valor", "rank_variable": "#Var", "rank_RADAR": "#RADAR",
        }),
        gradient_cols=["Valor"],
        cmap="RdYlGn_r" if reverse_color else "RdYlGn",
        fmt={"Valor": "{:,.2f}", "#Var": "{:.0f}", "#RADAR": "{:.0f}"},
    ).set_caption(f"{var_name}"))

    # Barplot
    plot_sorted = plot_df.sort_values(var_name, ascending=True)
    cscale = [[0, C["green"]], [0.5, C["gold"]], [1, C["red"]]] if reverse_color else [[0, C["gray_light"]], [1, C["gold"]]]
    fig = px.bar(plot_sorted, x=var_name, y="country_iso3", orientation="h",
                 title=title, color=var_name, color_continuous_scale=cscale)
    fig.update_layout(xaxis_title=xlabel, yaxis_title="", height=480, font=dict(family="Arial"))
    fig.show()

## 5. Variables por dimensión

In [34]:
display(Markdown("### D1 — Macroeconómica"))
for var in dim_vars_available.get("macro", []):
    label = variable_catalog.get(var, {}).get("description", var)
    direction = variable_catalog.get(var, {}).get("direction", "positive")
    barplot_variable(var, f"{label}", var.replace("_", " ").title(),
                     reverse_color=(direction == "negative"))

### D1 — Macroeconómica

**PIB nominal en USD corrientes** (↑ mayor = mejor)

,country_iso3,Valor,#Var,#RADAR
0,USA,30.99,15,6
1,CAN,28.44,14,14
2,MEX,28.25,13,9
3,ESP,28.18,12,3
4,CHL,26.52,11,5
5,PER,26.39,10,11
6,ECU,25.55,9,8
7,DOM,25.55,8,4
8,GTM,25.45,7,10
9,CRI,25.28,6,2


**PIB per capita PPP en dolares internacionales corrientes** (↑ mayor = mejor)

,country_iso3,Valor,#Var,#RADAR
0,USA,"85,809.90",15,6
1,CAN,"64,610.38",14,14
2,ESP,"57,965.29",13,3
3,PAN,"41,369.42",12,1
4,BHS,"41,197.93",11,15
5,URY,"36,417.87",10,7
6,CHL,"36,181.16",9,5
7,CRI,"31,106.76",8,2
8,DOM,"27,541.66",7,4
9,MEX,"26,185.36",6,9


...

,country_iso3,Valor,#Var,#RADAR
0,HND,91.11,15,13
1,SLV,84.66,14,12
2,PAN,83.67,13,1
3,BHS,79.24,12,15
4,MEX,74.59,11,9
5,CRI,71.33,10,2
6,ESP,69.95,9,3
7,CAN,65.15,8,14
8,CHL,63.86,7,5
9,ECU,57.20,6,8


**Inversión extranjera directa (% del PIB)** (↑ mayor = mejor)

,country_iso3,Valor,#Var,#RADAR
0,CRI,1.88,15,2
1,CHL,1.57,14,5
2,PAN,1.56,13,1
3,DOM,1.53,12,4
4,HND,1.51,11,13
5,CAN,1.34,10,14
6,SLV,1.28,9,12
7,ESP,1.25,8,3
8,MEX,1.24,7,9
9,PER,1.21,6,11


**Tarifa aplicada promedio ponderada para todos los productos** (↓ menor = mejor)

,country_iso3,Valor,#Var,#RADAR
0,BHS,16.34,1,15
1,PAN,5.93,2,1
2,ECU,5.19,3,8
3,URY,4.85,4,7
4,DOM,3.97,5,4
5,GTM,2.39,6,10
6,SLV,2.16,7,12
7,HND,2.11,8,13
8,MEX,1.62,9,9
9,USA,1.49,10,6


In [35]:
display(Markdown("### D2 — Industria Financiera"))
for var in dim_vars_available.get("financial", []):
    label = variable_catalog.get(var, {}).get("description", var)
    direction = variable_catalog.get(var, {}).get("direction", "positive")
    barplot_variable(var, f"{label}", var.replace("_", " ").title(),
                     reverse_color=(direction == "negative"))

### D2 — Industria Financiera

**Credito domestico al sector privado por bancos (% del PIB)** (↑ mayor = mejor)

,country_iso3,Valor,#Var,#RADAR
0,HND,4.35,15,13
1,CHL,4.33,14,5
2,ESP,4.30,13,3
3,PAN,4.24,12,1
4,ECU,4.02,11,8
5,SLV,3.99,10,12
6,CRI,3.95,9,2
7,USA,3.87,8,6
8,BHS,3.72,7,15
9,PER,3.71,6,11


**Adultos con cuenta en institucion financiera o servicio movil (%)** (↑ mayor = mejor)

,country_iso3,Valor,#Var,#RADAR
0,CAN,98.40,15,14
1,ESP,98.38,14,3
2,USA,97.02,13,6
3,CHL,85.07,12,5
4,URY,73.75,11,7
5,BHS,73.30,10,15
6,CRI,71.35,9,2
7,DOM,64.78,8,4
8,ECU,64.51,7,8
9,PAN,64.10,6,1


...

,country_iso3,Valor,#Var,#RADAR
0,URY,89.27,1,7
1,DOM,88.20,2,4
2,PER,87.59,3,11
3,SLV,87.27,4,12
4,ESP,85.05,5,3
5,CAN,84.76,6,14
6,GTM,81.02,7,10
7,HND,80.83,8,13
8,BHS,80.25,9,15
9,CHL,78.27,10,5


**Depositos del sistema financiero sobre PIB** (↑ mayor = mejor)

,country_iso3,Valor,#Var,#RADAR
0,ESP,4.79,15,3
1,USA,4.63,14,6
2,BHS,4.42,13,15
3,CAN,4.24,12,14
4,HND,4.18,11,13
5,CHL,4.15,10,5
6,SLV,4.07,9,12
7,PAN,4.07,8,1
8,URY,4.02,7,7
9,GTM,3.93,6,10


**Capital regulatorio bancario sobre activos ponderados por riesgo** (↑ mayor = mejor)

,country_iso3,Valor,#Var,#RADAR
0,MEX,17.70,15,9
1,ECU,17.68,13,8
2,URY,17.68,13,7
3,BHS,17.05,11,15
4,DOM,17.05,11,4
5,ESP,16.98,10,3
6,CRI,16.75,9,2
7,USA,16.28,8,6
8,GTM,16.24,7,10
9,CAN,16.10,6,14


In [36]:
display(Markdown("### D3 — Institucional"))
for var in dim_vars_available.get("institutional", []):
    label = variable_catalog.get(var, {}).get("description", var)
    direction = variable_catalog.get(var, {}).get("direction", "positive")
    barplot_variable(var, f"{label}", var.replace("_", " ").title(),
                     reverse_color=(direction == "negative"))

### D3 — Institucional

**WGI: calidad regulatoria** (↑ mayor = mejor)

,country_iso3,Valor,#Var,#RADAR
0,CAN,1.36,15,14
1,USA,1.35,14,6
2,CHL,0.78,13,5
3,URY,0.65,12,7
4,ESP,0.62,11,3
5,CRI,0.58,10,2
6,DOM,0.21,9,4
7,PER,0.08,8,11
8,PAN,-0.07,7,1
9,BHS,-0.11,6,15


**WGI: efectividad gubernamental** (↑ mayor = mejor)

,country_iso3,Valor,#Var,#RADAR
0,CAN,1.76,15,14
1,USA,1.36,14,6
2,ESP,1.11,13,3
3,CHL,0.96,12,5
4,BHS,0.71,11,15
5,URY,0.69,10,7
6,CRI,0.29,9,2
7,PAN,0.25,8,1
8,DOM,0.07,7,4
9,SLV,-0.18,6,12


...

,country_iso3,Valor,#Var,#RADAR
0,CAN,1.63,15,14
1,URY,1.53,14,7
2,BHS,1.23,13,15
3,CHL,1.11,12,5
4,USA,1.09,11,6
5,ESP,0.71,10,3
6,CRI,0.68,9,2
7,DOM,-0.27,8,4
8,PAN,-0.55,7,1
9,SLV,-0.58,6,12


**Prima de riesgo pais Damodaran / NYU Stern** (↓ menor = mejor)

,country_iso3,Valor,#Var,#RADAR
0,ECU,0.13,1,8
1,SLV,0.08,2,12
2,BHS,0.06,3,15
3,HND,0.06,3,13
4,CRI,0.04,5,2
5,DOM,0.04,5,4
6,GTM,0.03,7,10
7,PAN,0.03,8,1
8,MEX,0.02,9,9
9,PER,0.02,10,11


**Heritage Index of Economic Freedom** (↑ mayor = mejor)

,country_iso3,Valor,#Var,#RADAR
0,CAN,75.60,15,14
1,CHL,74.30,14,5
2,USA,72.80,13,6
3,URY,69.80,12,7
4,CRI,69.10,11,2
5,PER,66.30,10,11
6,BHS,65.10,9,15
7,PAN,64.90,8,1
8,DOM,63.80,7,4
9,ESP,63.65,6,3


In [37]:
display(Markdown("### D4 — Digital-Tecnológica"))
for var in dim_vars_available.get("digital_tech", []):
    label = variable_catalog.get(var, {}).get("description", var)
    direction = variable_catalog.get(var, {}).get("direction", "positive")
    barplot_variable(var, f"{label}", var.replace("_", " ").title(),
                     reverse_color=(direction == "negative"))

### D4 — Digital-Tecnológica

**Usuarios de internet (% de poblacion)** (↑ mayor = mejor)

,country_iso3,Valor,#Var,#RADAR
0,ESP,95.76,15,3
1,CHL,95.59,14,5
2,USA,94.69,13,6
3,CAN,94.35,12,14
4,BHS,92.47,11,15
5,URY,91.99,10,7
6,DOM,91.00,9,4
7,CRI,87.17,8,2
8,MEX,83.12,7,9
9,PER,81.96,6,11


**Suscripciones moviles por 100 habitantes** (↑ mayor = mejor)

,country_iso3,Valor,#Var,#RADAR
0,SLV,176.52,15,12
1,PAN,156.59,14,1
2,URY,145.50,13,7
3,CRI,136.02,12,2
4,CHL,132.66,11,5
5,ESP,130.26,10,3
6,PER,124.62,9,11
7,MEX,116.49,8,9
8,USA,113.19,7,6
9,GTM,112.52,6,10


...

,country_iso3,Valor,#Var,#RADAR
0,ESP,3.51,15,3
1,USA,3.32,14,6
2,GTM,3.17,13,10
3,BHS,3.12,12,15
4,CAN,3.00,11,14
5,PAN,2.97,10,1
6,CRI,2.82,9,2
7,HND,2.81,8,13
8,MEX,2.58,7,9
9,DOM,2.47,6,4


**Cajeros automáticos por cada 100.000 adultos** (↑ mayor = mejor)

,country_iso3,Valor,#Var,#RADAR
0,URY,5.76,15,7
1,CAN,5.24,14,14
2,USA,4.86,13,6
3,PER,4.86,12,11
4,BHS,4.77,11,15
5,ESP,4.48,10,3
6,PAN,4.30,9,1
7,MEX,4.24,8,9
8,CRI,4.16,7,2
9,CHL,3.88,6,5


**Exportaciones de TIC como porcentaje de las exportaciones totales** (↑ mayor = mejor)

,country_iso3,Valor,#Var,#RADAR
0,PAN,17.82,15,1
1,MEX,13.72,14,9
2,USA,7.85,13,6
3,CAN,1.37,11,14
4,ESP,1.37,11,3
5,CRI,0.83,10,2
6,BHS,0.71,9,15
7,DOM,0.56,8,4
8,SLV,0.40,7,12
9,HND,0.33,6,13


In [38]:
display(Markdown("### D5 — Proximidad y otras"))
for dim_key in sorted(dim_vars_available.keys()):
    if dim_key in ("macro", "financial", "institutional", "digital_tech"):
        continue
    for var in dim_vars_available[dim_key]:
        label = variable_catalog.get(var, {}).get("description", var)
        direction = variable_catalog.get(var, {}).get("direction", "positive")
        barplot_variable(var, f"{label}", var.replace("_", " ").title(),
                         reverse_color=(direction == "negative"))

### D5 — Proximidad y otras

**cultural_distance_hofstede** (↑ mayor = mejor)

,country_iso3,Valor,#Var,#RADAR
0,CAN,79.56,15,14
1,ESP,72.57,14,3
2,USA,70.79,13,6
3,CRI,58.42,12,2
4,URY,58.15,11,7
5,GTM,47.78,10,10
6,DOM,46.66,9,4
7,HND,45.10,8,13
8,BHS,45.02,7,15
9,CHL,44.82,6,5


**Distancia geografica a Bogota** (↓ menor = mejor)

,country_iso3,Valor,#Var,#RADAR
0,ESP,8.99,1,3
1,CAN,8.57,2,14
2,URY,8.54,3,7
3,CHL,8.35,4,5
4,USA,8.30,5,6
5,MEX,8.16,6,9
6,BHS,7.78,7,15
7,GTM,7.63,8,10
8,PER,7.54,9,11
9,SLV,7.53,10,12


...

,country_iso3,Valor,#Var,#RADAR
0,CAN,54.00,15,14
1,USA,50.00,14,6
2,ESP,47.00,13,3
3,URY,28.00,12,7
4,GTM,25.00,11,10
5,ECU,24.00,10,8
6,MEX,23.00,9,9
7,CRI,22.50,6,2
8,HND,22.50,6,13
9,PAN,22.50,6,1


**Hofstede: indulgence** (↑ mayor = mejor)

,country_iso3,Valor,#Var,#RADAR
0,MEX,97.00,15,9
1,CRI,89.00,10,2
2,GTM,89.00,10,10
3,HND,89.00,10,13
4,PAN,89.00,10,1
5,SLV,89.00,10,12
6,CAN,68.00,7,14
7,CHL,68.00,7,5
8,USA,68.00,7,6
9,BHS,67.00,6,15


**Salidas de colombianos hacia el pais destino** (↑ mayor = mejor)

,country_iso3,Valor,#Var,#RADAR
0,USA,12.45,15,6
1,ESP,11.70,14,3
2,PAN,10.94,13,1
3,MEX,10.72,12,9
4,ECU,10.68,11,8
5,CHL,10.58,10,5
6,DOM,10.32,9,4
7,CAN,9.94,8,14
8,PER,9.79,7,11
9,CRI,9.14,6,2


## 6. Fichas top-5: dimensión más fuerte y más débil

In [40]:
if dim_available:
    for country in top_5:
        if country not in global_ranking["country_iso3"].values:
            continue
        row = global_ranking[global_ranking["country_iso3"] == country].iloc[0]
        rank_pos = int(row.get("rank", 0))
        radar_score = radar_global[radar_global["country_iso3"] == country]["radar_score"].values
        radar_val = f"{radar_score[0]:.3f}" if len(radar_score) > 0 else "N/A"

        dim_scores = {DIM_LABELS_SHORT[DIM_COLS.index(c)]: row[c] for c in dim_available if c in row.index}
        strongest = max(dim_scores, key=dim_scores.get)
        weakest = min(dim_scores, key=dim_scores.get)

        display(Markdown(f"### {country} — #{rank_pos} TOPSIS | RADAR: {radar_val}"))
        ficha = pd.DataFrame([{
            "Dimensión": k, "Score": f"{v:.3f}",
            "Rol": "⬆ Fortaleza" if k == strongest else ("⬇ Debilidad" if k == weakest else "—"),
        } for k, v in dim_scores.items()])
        display(style_table(ficha))

### PAN — #8 TOPSIS | RADAR: 0.691

,Dimensión,Score,Rol
0,Macro,0.459,⬇ Debilidad
1,Financiera,0.528,—
2,Institucional,0.543,⬆ Fortaleza
3,Digital-Tech,0.532,—


### CRI — #6 TOPSIS | RADAR: 0.653

,Dimensión,Score,Rol
0,Macro,0.451,⬇ Debilidad
1,Financiera,0.481,—
2,Institucional,0.709,⬆ Fortaleza
3,Digital-Tech,0.547,—


### ESP — #3 TOPSIS | RADAR: 0.627

,Dimensión,Score,Rol
0,Macro,0.604,⬇ Debilidad
1,Financiera,0.604,—
2,Institucional,0.725,⬆ Fortaleza
3,Digital-Tech,0.712,—


### DOM — #11 TOPSIS | RADAR: 0.620

,Dimensión,Score,Rol
0,Macro,0.448,—
1,Financiera,0.492,—
2,Institucional,0.559,⬆ Fortaleza
3,Digital-Tech,0.437,⬇ Debilidad


### CHL — #4 TOPSIS | RADAR: 0.569

,Dimensión,Score,Rol
0,Macro,0.519,⬇ Debilidad
1,Financiera,0.554,—
2,Institucional,0.763,⬆ Fortaleza
3,Digital-Tech,0.645,—


## 7. Scatter estratégico: score institucional vs score financiero

Los ejes representan las dos dimensiones con mayor peso en el modelo (30% cada una).
El tamaño del punto es proporcional al score RADAR compuesto.

In [43]:
if "score_institutional" in global_ranking.columns and "score_financial" in global_ranking.columns:
    scatter_df = global_ranking[global_ranking["country_iso3"].isin(top_15)].merge(
        radar_global[["country_iso3", "radar_score"]], on="country_iso3", how="left"
    )
    fig = px.scatter(
        scatter_df, x="score_institutional", y="score_financial",
        text="country_iso3", size="radar_score", color="radar_score",
        color_continuous_scale=[[0, C["gray_light"]], [0.5, C["gold"]], [1, C["green"]]],
        title="Institucionalidad vs profundidad financiera: los dos pilares del atractivo estructural",
    )
    fig.update_traces(textposition="top center", textfont_size=10)

    # Líneas de cuadrante en 0.5
    fig.add_hline(y=0.5, line_dash="dash", line_color=C["gray_mid"], line_width=1.5, opacity=0.6)
    fig.add_vline(x=0.5, line_dash="dash", line_color=C["gray_mid"], line_width=1.5, opacity=0.6)

    # Etiquetas de cuadrante
    fig.add_annotation(x=0.85, y=0.92, text="Mercado maduro",
                       showarrow=False, font=dict(size=11, color=C["green"]), opacity=0.7)
    fig.add_annotation(x=0.15, y=0.92, text="Financiero pero\ninstitución débil",
                       showarrow=False, font=dict(size=10, color=C["amber"]), opacity=0.7)
    fig.add_annotation(x=0.85, y=0.08, text="Institucional pero\nfinancieramente limitado",
                       showarrow=False, font=dict(size=10, color=C["amber"]), opacity=0.7)
    fig.add_annotation(x=0.15, y=0.08, text="Corredor / Emergente",
                       showarrow=False, font=dict(size=11, color=C["red"]), opacity=0.7)

    fig.update_layout(
        xaxis_title="Score institucional (WGI + Prima de Riesgo Pais + Libertad económica)",
        yaxis_title="Score industria financiera",
        xaxis=dict(range=[0, 1]), yaxis=dict(range=[0, 1]),
        height=580, font=dict(family="Arial"),
    )
    fig.show()

    insight_box("Cuatro cuadrantes",
        "Arriba-derecha: mercados maduros con institucionalidad y profundidad (USA, CAN, CHL, ESP). "
        "Abajo-izquierda: mercados de corredor donde la oportunidad viene de proximidad. "
        "Los cuadrantes mixtos señalan mercados con una fortaleza aprovechable pero una restricción crítica.")

## 8. Scatter integral: entorno completo vs profundidad financiera

El eje X combina las cuatro dimensiones no financieras (macro, institucional,
digital-tecnológica y proximidad) en un promedio simple. El eje Y es la
dimensión financiera. Esto permite ver el comportamiento de los cinco
componentes del radar en un solo plano.

In [ ]:
env_dims = [c for c in ["score_macro", "score_institutional", "score_digital_tech"] if c in global_ranking.columns]
if len(env_dims) >= 2 and "score_financial" in global_ranking.columns:
   scatter2 = global_ranking[global_ranking["country_iso3"].isin(top_15)].copy()
   # Promedio de dimensiones estructurales no financieras
   scatter2["entorno_estructural"] = scatter2[env_dims].mean(axis=1)
   # Agregar IPC si disponible
   if has_ipc:
       scatter2["ipc_val"] = scatter2["country_iso3"].map(ipc_map)
       # Promedio ponderado: 75% dimensiones TOPSIS + 25% proximidad (reparto 5% de la tendencia (10%) a cada factor
       scatter2["entorno_integral"] = 0.75 * scatter2["entorno_estructural"] + 0.25 * scatter2["ipc_val"].fillna(0)
       x_col = "entorno_integral"
       x_label = "Entorno integral (Macro + Institucional + Digital + Proximidad)"
   else:
       x_col = "entorno_estructural"
       x_label = "Entorno estructural (Macro + Institucional + Digital)"
   scatter2 = scatter2.merge(
       radar_global[["country_iso3", "radar_score"]], on="country_iso3", how="left"
   )
   fig = px.scatter(
       scatter2, x=x_col, y="score_financial",
       text="country_iso3", size="radar_score", color="radar_score",
       color_continuous_scale=[[0, C["gray_light"]], [0.5, C["gold"]], [1, C["green"]]],
       title="Entorno integral vs profundidad financiera: la vista completa del radar en un plano",
   )
   fig.update_traces(textposition="top center", textfont_size=10)
   # Líneas de cuadrante en 0.5
   fig.add_hline(y=0.5, line_dash="dash", line_color=C["gray_mid"], line_width=1.5, opacity=0.6)
   fig.add_vline(x=0.5, line_dash="dash", line_color=C["gray_mid"], line_width=1.5, opacity=0.6)
   # Etiquetas de cuadrante
   fig.add_annotation(x=0.85, y=0.92, text="Atractivo integral",
                      showarrow=False, font=dict(size=11, color=C["green"]), opacity=0.7)
   fig.add_annotation(x=0.15, y=0.92, text="Financiero pero\nentorno débil",
                      showarrow=False, font=dict(size=10, color=C["amber"]), opacity=0.7)
   fig.add_annotation(x=0.85, y=0.08, text="Buen entorno pero\nsin profundidad financiera",
                      showarrow=False, font=dict(size=10, color=C["amber"]), opacity=0.7)
   fig.add_annotation(x=0.15, y=0.08, text="Baja atractividad\nestructural",
                      showarrow=False, font=dict(size=11, color=C["red"]), opacity=0.7)
   fig.update_layout(
       xaxis_title=x_label,
       yaxis_title="Score industria financiera",
       xaxis=dict(range=[0, 1]), yaxis=dict(range=[0, 1]),
       height=580, font=dict(family="Arial"),
   )
   fig.show()
   # Tabla de datos del scatter
   scatter2_display = scatter2[["country_iso3", x_col, "score_financial", "radar_score"]].sort_values("radar_score", ascending=False)
   scatter2_display = scatter2_display.rename(columns={
       x_col: "Entorno", "score_financial": "Financiera", "radar_score": "RADAR",
   })
   display(style_table(
       scatter2_display,
       gradient_cols=["Entorno", "Financiera", "RADAR"], cmap="RdYlGn",
       fmt={"Entorno": "{:.3f}", "Financiera": "{:.3f}", "RADAR": "{:.3f}"},
   ).set_caption("Datos del scatter integral"))
   insight_box("Lectura integral",
       "Este gráfico comprime las 5 dimensiones del radar en 2 ejes. "
       "Los países arriba-derecha combinan entorno favorable con sistema financiero profundo. "
       "Los países que suben por proximidad (PAN, CRI) se desplazan a la derecha respecto al scatter anterior. "
       "La diferencia entre ambos scatters revela cuánto aporta la proximidad a cada país.")

,country_iso3,Entorno,Financiera,RADAR
7,PAN,0.621,0.528,0.691
5,CRI,0.625,0.481,0.653
2,ESP,0.510,0.604,0.627
8,DOM,0.559,0.492,0.620
3,CHL,0.548,0.554,0.569
0,USA,0.620,0.648,0.552
4,URY,0.515,0.493,0.539
13,ECU,0.509,0.510,0.538
9,MEX,0.452,0.461,0.533
12,GTM,0.407,0.469,0.530


In [ ]:
# Descubrir variables disponibles y mapearlas a dimensiones
available_vars = [c for c in wide_raw.columns if c != "country_iso3"]
print(f"\n{len(available_vars)} variables disponibles en wide_raw:")

dim_vars_available: Dict[str, List[str]] = {}
for var in available_vars:
    meta = variable_catalog.get(var, {})
    dim = meta.get("dimension", "other")
    dim_vars_available.setdefault(dim, []).append(var)
    
for dim, vars_list in sorted(dim_vars_available.items()):
    print(f"  {dim}: {', '.join(vars_list)}")


12 variables disponibles en wide_raw:
  institutional: country_risk_premium, heritage_efi
  other: cultural_distance_hofstede
  proximity: geographic_distance_km, common_language_spanish, hofstede_pdi, hofstede_idv, hofstede_mas, hofstede_uai, hofstede_lto, hofstede_ivr, colombian_diaspora_stock


## Síntesis Ejecutiva

Los scores dimensionales revelan que no todos los países llegan al ranking por las
mismas razones. Los mercados maduros (USA, CAN, CHL, ESP) dominan en institucionalidad
y profundidad financiera. Los mercados de corredor (PAN, CRI, DOM, ECU) compensan
debilidad estructural con proximidad. El radar chart hace visible esta distinción
con más claridad que un ranking plano.